# Football Lake — Huấn luyện LightGBM theo tháng

Notebook này dùng `mart_ml_training_snapshot.parquet`, thay cho việc tự chia lại
`mart_ml_match_features` bằng hai ngày cố định. Mặc định dbt chốt tháng UTC đã hoàn tất,
giữ một tháng Validation và một tháng Test; Train tích lũy toàn bộ lịch sử trước đó.

Trước khi chạy: dựng snapshot dbt thành công, kết nối mạng tới MinIO/MLflow và cài các
thư viện vốn dùng trong notebook gốc (`duckdb`, `pandas`, `numpy`, `lightgbm`,
`scikit-learn`, `matplotlib`, `mlflow`). Có thể giữ các cell thiết lập Tailscale của bạn.

Cell đầu đọc cấu hình từ biến môi trường hoặc Colab Secrets. Không điền mật khẩu
hay token vào mã nguồn. `MLFLOW_TRACKING_URI` có thể đặt để đổi máy chủ Tracking.

Cell huấn luyện giữ cấu hình LightGBM 500 vòng, learning rate 0,03, num_leaves 15,
min_child_samples 40 và trọng số hòa 1,5 của notebook đã gửi. Danh sách 92 đặc trưng
của phiên bản 2 là tập ứng viên; việc lọc thiếu/biến thiên chỉ dùng Train, nên số cột
thực tế có thể ít hơn. Các biến ngoài danh sách này không tự được bật lại.

Kết quả là một run ứng viên có ngày chốt, mốc chia tập và fingerprint dữ liệu.
Notebook không tự đổi alias champion và không thay mô hình đang phục vụ dashboard.
Các chỉ số trong notebook dùng argmax; quy tắc ngưỡng hòa trên VM là quy tắc riêng.


In [ ]:
import os
from getpass import getpass
import duckdb

def setting(name, default=None, secret=False):
    value = os.getenv(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            pass
    if not value:
        value = default
    if not value:
        value = getpass(name + ": ") if secret else input(name + ": ").strip()
    if not value:
        raise ValueError("Thiếu cấu hình " + name)
    return value

endpoint = setting("MINIO_ENDPOINT")
use_ssl = endpoint.startswith("https://")
endpoint = endpoint.removeprefix("http://").removeprefix("https://").rstrip("/")
bucket = setting("MINIO_BUCKET", "football-lake")
access_key = setting("MINIO_ACCESS_KEY", secret=True)
secret_key = setting("MINIO_SECRET_KEY", secret=True)

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

con = duckdb.connect(":memory:")
try:
    con.execute("INSTALL httpfs; LOAD httpfs;")
    for key, value in {
        "s3_endpoint": endpoint,
        "s3_access_key_id": access_key,
        "s3_secret_access_key": secret_key,
        "s3_url_style": "path",
    }.items():
        con.execute("SET " + key + " = " + sql_literal(value))
    con.execute("SET s3_use_ssl = " + ("true" if use_ssl else "false"))
    source_path = f"s3://{bucket}/dwh/mart_ml_training_snapshot.parquet"
    df = con.execute("SELECT * FROM read_parquet(?)", [source_path]).df()
finally:
    con.close()
print("Đã đọc snapshot:", len(df), "trận")


In [ ]:
"""Validate the dbt monthly snapshot before fitting any estimator."""

import hashlib
import json
import numpy as np
import pandas as pd

META = {
    'window_id': 'training_window_id',
    'split_policy': 'training_split_policy',
    'data_as_of': 'training_data_as_of',
    'train_end': 'training_train_end',
    'valid_start': 'training_valid_start',
    'valid_end': 'training_valid_end',
    'test_start': 'training_test_start',
    'valid_months': 'training_valid_months',
    'test_months': 'training_test_months',
    'generated_at': 'training_generated_at',
}


def _utc_naive(values):
    return pd.to_datetime(values, errors='raise', utc=True).dt.tz_convert(None)


def validate_monthly_snapshot(df: pd.DataFrame) -> dict:
    required = list(META.values()) + [
        'match_key', 'match_date', 'kickoff_utc', 'split', 'is_warm',
        'is_labeled', 'is_finished', 'y_result_code',
        'training_label_known_ts', 'training_label_cutoff_ts',
    ]
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError('Không phải snapshot theo tháng hoặc thiếu cột: ' + ', '.join(missing))
    if df.empty:
        raise ValueError('Snapshot rỗng; kiểm tra nguồn dữ liệu và ngày chốt.')

    meta = {}
    for name, column in META.items():
        if df[column].isna().any() or df[column].nunique(dropna=False) != 1:
            raise ValueError(f'Metadata không đồng nhất: {column}')
        meta[name] = df[column].iloc[0]
    if meta['split_policy'] != 'monthly_v1':
        raise ValueError('Phiên bản chính sách chia dữ liệu chưa được hỗ trợ.')

    for name in ['data_as_of', 'train_end', 'valid_start', 'valid_end', 'test_start']:
        meta[name] = pd.Timestamp(meta[name]).normalize()
    for name in ['valid_months', 'test_months']:
        value = meta[name]
        if isinstance(value, (bool, np.bool_)) or int(value) != value or int(value) < 1:
            raise ValueError(f'{name} phải là số nguyên dương.')
        meta[name] = int(value)

    as_of = meta['data_as_of']
    expected_test_start = as_of.replace(day=1) - pd.DateOffset(months=meta['test_months']-1)
    expected_valid_start = expected_test_start - pd.DateOffset(months=meta['valid_months'])
    expected = {
        'test_start': expected_test_start,
        'valid_end': expected_test_start - pd.Timedelta(days=1),
        'valid_start': expected_valid_start,
        'train_end': expected_valid_start - pd.Timedelta(days=1),
    }
    if any(meta[name] != value for name, value in expected.items()):
        raise ValueError('Các mốc chia tập không đúng chính sách theo tháng.')
    generated = pd.to_datetime(meta['generated_at'], utc=True).tz_convert(None)
    if as_of >= generated.normalize():
        raise ValueError('Ngày chốt phải trước ngày tạo snapshot, không được dùng ngày tương lai.')
    expected_id = hashlib.md5(
        f"monthly_v1|{as_of.date()}|{meta['valid_months']}|{meta['test_months']}".encode()
    ).hexdigest()
    if meta['window_id'] != expected_id:
        raise ValueError('window_id không khớp với các mốc thời gian.')

    if df['match_key'].isna().any() or df['match_key'].duplicated().any():
        raise ValueError('match_key phải khác null và không trùng.')
    dates = _utc_naive(df['match_date']).dt.normalize()
    if dates.isna().any() or (dates > as_of).any():
        raise ValueError('Có trận thiếu ngày hoặc vượt ngày chốt dữ liệu.')
    for column in ['is_warm', 'is_labeled', 'is_finished']:
        if not df[column].eq(True).fillna(False).all():
            raise ValueError(f'Snapshot chứa bản ghi không đủ điều kiện: {column}')
    labels = pd.to_numeric(df['y_result_code'], errors='coerce')
    if not labels.isin([0, 1, 2]).all():
        raise ValueError('Snapshot chứa nhãn kết quả thiếu hoặc không hợp lệ.')

    if df['split'].isna().any() or not df['split'].isin(['train', 'valid', 'test']).all():
        raise ValueError('Snapshot chứa split thiếu hoặc không hợp lệ.')
    expected_split = np.select(
        [dates <= meta['train_end'], dates <= meta['valid_end']],
        ['train', 'valid'], default='test',
    )
    if not np.array_equal(df['split'].to_numpy(), expected_split):
        raise ValueError('split từ dbt không khớp với các mốc trong metadata; không tự ghi đè split.')
    for split in ['train', 'valid', 'test']:
        if not df['split'].eq(split).any():
            raise ValueError(f'Tập {split} rỗng; cần kiểm tra độ phủ dữ liệu.')
    if set(labels[df['split'].eq('train')].unique()) != {0, 1, 2}:
        raise ValueError('Train phải có đủ ba lớp kết quả.')

    known = _utc_naive(df['training_label_known_ts'])
    kickoff = _utc_naive(df['kickoff_utc'])
    expected_known = (kickoff + pd.Timedelta(hours=3)).fillna(
        dates + pd.Timedelta(hours=23, minutes=59, seconds=59)
    )
    cutoff = _utc_naive(df['training_label_cutoff_ts'])
    expected_cutoff = pd.Series(as_of + pd.Timedelta(days=1), index=df.index)
    expected_cutoff.loc[df['split'].eq('train')] = meta['valid_start']
    expected_cutoff.loc[df['split'].eq('valid')] = meta['test_start']
    if known.isna().any() or not known.eq(expected_known).all():
        raise ValueError('Mốc khả dụng của nhãn không khớp với chính sách kickoff + 3 giờ / cuối ngày.')
    if cutoff.isna().any() or not cutoff.eq(expected_cutoff).all() or not known.lt(cutoff).all():
        raise ValueError('Có kết quả trận chưa khả dụng tại mốc chia tập tương ứng.')

    # Exclude build metadata from the content fingerprint; row order is immaterial.
    columns = sorted(c for c in df.columns if not c.startswith('training_'))
    canonical = df.sort_values('match_key').reset_index(drop=True)[columns]
    h = hashlib.sha256()
    h.update(json.dumps(columns, ensure_ascii=False).encode())
    h.update(pd.util.hash_pandas_object(canonical, index=False).values.tobytes())
    result = {
        key: value.date().isoformat() if isinstance(value, pd.Timestamp) else value
        for key, value in meta.items() if key != 'generated_at'
    }
    result.update({
        'generated_at': generated.isoformat(),
        'snapshot_sha256': h.hexdigest(),
        'pandas_version': pd.__version__,
        'split_counts': {key: int(value) for key, value in df['split'].value_counts().items()},
        'label_availability_policy': 'kickoff_plus_3h_or_end_of_day_estimate',
    })
    return result


def select_train_features(train_df, candidates, max_missing_rate=0.15):
    if not 0 <= max_missing_rate < 1:
        raise ValueError('max_missing_rate phải nằm trong [0, 1).')
    candidates = list(dict.fromkeys(candidates))
    if any(not c.startswith('f_') or c not in train_df.columns for c in candidates):
        raise ValueError('Chỉ được dùng các cột đặc trưng f_ trong danh sách cho phép.')
    x = train_df[candidates].replace([np.inf, -np.inf], np.nan)
    missing = x.isna().mean()
    nunique = x.nunique(dropna=True)
    features = [c for c in candidates if missing[c] <= max_missing_rate and nunique[c] > 1]
    if not features:
        raise ValueError('Không còn đặc trưng sau khi lọc trên Train.')
    return features, missing, nunique


MONTHLY_FEATURE_ALLOWLIST = [
  "f_home_n_prev",
  "f_away_n_prev",
  "f_diff_n_prev",
  "f_home_pts_l3",
  "f_away_pts_l3",
  "f_diff_pts_l3",
  "f_home_gf_l3",
  "f_away_gf_l3",
  "f_diff_gf_l3",
  "f_home_ga_l3",
  "f_away_ga_l3",
  "f_diff_ga_l3",
  "f_home_pts_l5",
  "f_away_pts_l5",
  "f_diff_pts_l5",
  "f_home_gf_l5",
  "f_away_gf_l5",
  "f_diff_gf_l5",
  "f_home_ga_l5",
  "f_away_ga_l5",
  "f_diff_ga_l5",
  "f_home_gd_l5",
  "f_away_gd_l5",
  "f_diff_gd_l5",
  "f_home_win_rate_l5",
  "f_away_win_rate_l5",
  "f_diff_win_rate_l5",
  "f_home_cs_rate_l5",
  "f_away_cs_rate_l5",
  "f_diff_cs_rate_l5",
  "f_home_btts_rate_l5",
  "f_away_btts_rate_l5",
  "f_diff_btts_rate_l5",
  "f_home_over25_rate_l5",
  "f_away_over25_rate_l5",
  "f_diff_over25_rate_l5",
  "f_home_pts_l10",
  "f_away_pts_l10",
  "f_diff_pts_l10",
  "f_home_gf_l10",
  "f_away_gf_l10",
  "f_diff_gf_l10",
  "f_home_ga_l10",
  "f_away_ga_l10",
  "f_diff_ga_l10",
  "f_home_win_rate_l10",
  "f_away_win_rate_l10",
  "f_diff_win_rate_l10",
  "f_home_side_pts_l5",
  "f_away_side_pts_l5",
  "f_diff_side_pts_l5",
  "f_home_side_gf_l5",
  "f_away_side_gf_l5",
  "f_diff_side_gf_l5",
  "f_home_side_ga_l5",
  "f_away_side_ga_l5",
  "f_diff_side_ga_l5",
  "f_home_s_played",
  "f_away_s_played",
  "f_diff_s_played",
  "f_home_s_ppg",
  "f_away_s_ppg",
  "f_diff_s_ppg",
  "f_home_s_gf_pg",
  "f_away_s_gf_pg",
  "f_diff_s_gf_pg",
  "f_home_s_ga_pg",
  "f_away_s_ga_pg",
  "f_diff_s_ga_pg",
  "f_home_s_gd_pg",
  "f_away_s_gd_pg",
  "f_diff_s_gd_pg",
  "f_h2h_n",
  "f_home_rest_days",
  "f_away_rest_days",
  "f_diff_rest_days",
  "f_home_n_7d",
  "f_away_n_7d",
  "f_diff_n_7d",
  "f_home_n_14d",
  "f_away_n_14d",
  "f_diff_n_14d",
  "f_home_n_30d",
  "f_away_n_30d",
  "f_diff_n_30d",
  "f_dow",
  "f_month",
  "f_is_weekend",
  "f_kickoff_hour_utc",
  "f_ref_n_prev",
  "f_home_form5_points_rate",
  "f_away_form5_points_rate"
]


In [ ]:

# ============================================================
# FOOTBALL 1X2 MULTICLASS - LIGHTGBM + MLFLOW
# Classes: 0 = Away Win, 1 = Draw, 2 = Home Win
#
# Input:
#   df: DataFrame da doc tu dbt / DuckDB
#
# Pipeline:
#   1. Validate input
#   2. Filter labeled + warm matches
#   3. Calendar-based train/valid/test split
#   4. Feature selection based ONLY on train
#   5. Train LightGBM with Draw sample weight
#   6. Evaluate validation and test
#   7. Save reports, predictions, confusion matrices,
#      feature importance, model and metadata to MLflow
# ============================================================

import os
import json
import tempfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.lightgbm

from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    matthews_corrcoef,
    log_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATE_COL = "match_date"
TARGET_COL = "y_result_code"
KEY_COL = "match_key"

LABELS = [0, 1, 2]
CLASS_NAMES = ["Away win", "Draw", "Home win"]
CLASS_MAPPING = {
    0: "Away win",
    1: "Draw",
    2: "Home win",
}

# Read and validate the boundaries emitted by dbt. Never rebuild split locally.
MONTHLY_WINDOW = validate_monthly_snapshot(df)
TRAIN_END = pd.Timestamp(MONTHLY_WINDOW["train_end"])
VALID_END = pd.Timestamp(MONTHLY_WINDOW["valid_end"])
DATA_AS_OF = pd.Timestamp(MONTHLY_WINDOW["data_as_of"])

MAX_MISSING_RATE = 0.15

# Keep fixed for the first comparable baseline.
DRAW_WEIGHT = 1.5

RANDOM_STATE = 42

EXPERIMENT_NAME = "Football_LightGBM_Tuning"
RUN_NAME = "LightGBM_Monthly_" + DATA_AS_OF.strftime("%Y_%m_%d")

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://100.116.34.36:8080")

MODEL_PARAMS = {
    "objective": "multiclass",
    "num_class": 3,
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 15,
    "min_child_samples": 40,
    "reg_lambda": 1.0,
    "colsample_bytree": 0.85,
    "subsample": 0.85,
    "subsample_freq": 1,
    "random_state": RANDOM_STATE,
    "verbosity": -1,
    "n_jobs": -1,
}


# ============================================================
# 2. CHECK INPUT DATA
# ============================================================

if "df" not in globals():
    raise NameError(
        "Không tìm thấy DataFrame `df`. "
        "Hãy load dữ liệu từ dbt/DuckDB trước khi chạy."
    )

required_cols = [
    DATE_COL,
    TARGET_COL,
    "is_warm",
    "is_labeled",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Thiếu cột bắt buộc: {missing_cols}")

data = df.copy()
n_raw = len(data)

# Normalize date to calendar day for splitting.
data[DATE_COL] = pd.to_datetime(
    data[DATE_COL], errors="coerce"
)

# Strictly validate target labels before filtering invalid rows.
target_numeric = pd.to_numeric(
    data[TARGET_COL], errors="coerce"
)

target_present = target_numeric.notna()
invalid_label = target_present & ~target_numeric.isin(LABELS)

if invalid_label.any():
    bad_values = sorted(
        target_numeric.loc[invalid_label].unique().tolist()
    )
    raise ValueError(
        f"{TARGET_COL} chứa nhãn ngoài {LABELS}: {bad_values}"
    )

# Convert boolean-like columns only if they are actual booleans.
# Avoid treating the string 'False' as True.
def as_true_mask(series):
    if pd.api.types.is_bool_dtype(series):
        return series.eq(True)

    normalized = series.astype("string").str.strip().str.lower()
    return normalized.isin(["true", "1", "yes", "y"])

warm_mask = as_true_mask(data["is_warm"])
labeled_mask = as_true_mask(data["is_labeled"])

keep_mask = (
    data[DATE_COL].notna()
    & warm_mask
    & labeled_mask
    & target_present
)

data = data.loc[keep_mask].copy()
data[TARGET_COL] = target_numeric.loc[data.index].astype("int8")

if data.empty:
    raise ValueError(
        "Không còn dữ liệu sau khi lọc ngày, is_warm, "
        "is_labeled và target."
    )

# The target must match the declared business mapping.
# This check validates codes, not their semantic meaning.
observed_labels = set(data[TARGET_COL].unique())
if not observed_labels.issubset(set(LABELS)):
    raise ValueError(f"Nhãn không hợp lệ: {observed_labels}")


# ============================================================
# 3. CHECK MATCH UNIQUENESS
# ============================================================

if KEY_COL in data.columns:
    if data[KEY_COL].isna().any():
        raise ValueError("Có match_key null trong dữ liệu được giữ.")

    duplicate_keys = data[KEY_COL].duplicated(keep=False)

    if duplicate_keys.any():
        examples = data.loc[
            duplicate_keys, [KEY_COL, DATE_COL, TARGET_COL]
        ].head(20)

        raise ValueError(
            "Một match_key có nhiều hơn một dòng. "
            "Cần xử lý duplicate trước khi train.\n"
            + examples.to_string(index=False)
        )


# ============================================================
# 4. REBUILD SPLIT BY CALENDAR DATE
# ============================================================

calendar_date = data[DATE_COL].dt.normalize()

# The split is owned by dbt; validation above rejects a mismatched export.
assert (calendar_date <= DATA_AS_OF).all()

split_order = ["train", "valid", "test"]

split_counts = (
    data["split"]
    .value_counts()
    .reindex(split_order, fill_value=0)
)

print("=" * 70)
print("DATA AUDIT")
print("=" * 70)
print("Rows before filtering:", n_raw)
print("Rows after filtering:", len(data))
print("\nRows per split:")
print(split_counts)

if (split_counts == 0).any():
    raise ValueError(
        "Có tập train/valid/test rỗng. "
        "Kiểm tra DATE_COL và các cutoff."
    )

# All matches on the same calendar date must belong to one split.
date_split_counts = (
    data.assign(_calendar_date=calendar_date)
    .groupby("_calendar_date")["split"]
    .nunique()
)

if (date_split_counts > 1).any():
    raise AssertionError(
        "Phát hiện ngày lịch bị chia sang nhiều split."
    )

# Verify the actual temporal boundaries.
train_df = data.loc[data["split"].eq("train")].copy()
valid_df = data.loc[data["split"].eq("valid")].copy()
test_df = data.loc[data["split"].eq("test")].copy()

assert train_df[DATE_COL].max().normalize() <= TRAIN_END
assert valid_df[DATE_COL].min().normalize() > TRAIN_END
assert valid_df[DATE_COL].max().normalize() <= VALID_END
assert test_df[DATE_COL].min().normalize() > VALID_END
assert test_df[DATE_COL].max().normalize() <= DATA_AS_OF

print("\nDate range by split:")
for split_name, part in [
    ("train", train_df),
    ("valid", valid_df),
    ("test", test_df),
]:
    print(
        f"{split_name:>5}: "
        f"{part[DATE_COL].min()} -> {part[DATE_COL].max()}"
    )


# ============================================================
# 5. CHECK LABEL DISTRIBUTION
# ============================================================

print("\nLabel distribution:")
for split_name, part in [
    ("train", train_df),
    ("valid", valid_df),
    ("test", test_df),
]:
    print(f"\n{split_name.upper()} (n={len(part)}):")

    distribution = (
        part[TARGET_COL]
        .value_counts(normalize=True)
        .reindex(LABELS, fill_value=0)
    )

    distribution.index = [
        CLASS_MAPPING[i] for i in distribution.index
    ]

    print(distribution.round(4))

    absent = [
        label for label in LABELS
        if label not in set(part[TARGET_COL].unique())
    ]

    if absent:
        print(f"WARNING: {split_name} thiếu nhãn {absent}")

# Training must contain all three classes.
train_classes = np.sort(train_df[TARGET_COL].unique())

if not np.array_equal(train_classes, np.array(LABELS)):
    raise ValueError(
        "Train phải có đủ ba lớp [0, 1, 2]. "
        f"Hiện có {train_classes.tolist()}. "
        "Kiểm tra cutoff hoặc dữ liệu nhãn."
    )


# ============================================================
# 6. ENCODE FORM FEATURES
# ============================================================

def form_to_points(form):
    """
    W = 3 points, D = 1 point, L = 0 points.
    Return points / maximum possible points, in [0, 1].
    """
    if not isinstance(form, str) or not form.strip():
        return np.nan

    score_map = {"W": 3, "D": 1, "L": 0}
    values = [
        score_map[c]
        for c in form.upper()
        if c in score_map
    ]

    if not values:
        return np.nan

    return float(sum(values) / (3 * len(values)))


for col in ["f_home_form5", "f_away_form5"]:
    if col in data.columns:
        data[col + "_points_rate"] = data[col].map(
            form_to_points
        )

# Refresh split frames because new encoded columns were added.
train_df = data.loc[data["split"].eq("train")].copy()
valid_df = data.loc[data["split"].eq("valid")].copy()
test_df = data.loc[data["split"].eq("test")].copy()


# ============================================================
# 7. FEATURE SELECTION - TRAIN ONLY
# ============================================================

# Use numeric f_ columns only.
# Target columns (y_), identifiers, split, dates, and raw form strings
# are not eligible as predictors.
features_all = [
    col
    for col in data.columns
    if col in MONTHLY_FEATURE_ALLOWLIST
    and col.startswith("f_")
    and pd.api.types.is_numeric_dtype(data[col])
]

features_all = list(dict.fromkeys(features_all))

if not features_all:
    raise ValueError(
        "Không tìm thấy feature f_ dạng số."
    )

features, missing_rate_train, nunique_train = select_train_features(
    train_df, features_all, MAX_MISSING_RATE
)

if not features:
    raise ValueError(
        "Không còn feature sau lọc missing/variance. "
        "Hãy kiểm tra data hoặc MAX_MISSING_RATE."
    )

# Report missingness by split for selected features.
missing_report = pd.DataFrame({
    "train": train_df[features].isna().mean(),
    "valid": valid_df[features].isna().mean(),
    "test": test_df[features].isna().mean(),
})

missing_report["max_split_gap"] = (
    missing_report[["train", "valid", "test"]].max(axis=1)
    - missing_report[["train", "valid", "test"]].min(axis=1)
)

print("\nFeature selection:")
print("Numeric feature candidates:", len(features_all))
print("Selected features:", len(features))
print("Train missing threshold:", MAX_MISSING_RATE)

print("\nTop 15 selected features with largest missing-rate gap:")
print(
    missing_report.sort_values(
        "max_split_gap", ascending=False
    ).head(15).round(3)
)

# Build final matrices with the same feature order.
X_train = train_df[features].copy()
X_valid = valid_df[features].copy()
X_test = test_df[features].copy()

for X in [X_train, X_valid, X_test]:
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    for col in features:
        X[col] = pd.to_numeric(X[col], errors="coerce").astype("float32")

y_train = train_df[TARGET_COL].astype("int8").copy()
y_valid = valid_df[TARGET_COL].astype("int8").copy()
y_test = test_df[TARGET_COL].astype("int8").copy()

feature_names = list(X_train.columns)

if list(X_valid.columns) != feature_names:
    raise ValueError("Feature validation không khớp train.")

if list(X_test.columns) != feature_names:
    raise ValueError("Feature test không khớp train.")

if len(set(feature_names)) != len(feature_names):
    raise ValueError("Tên feature bị trùng.")

# Confirm feature matrices contain no infinite values.
for split_name, X in [
    ("train", X_train),
    ("valid", X_valid),
    ("test", X_test),
]:
    if np.isinf(X.to_numpy(dtype=np.float64)).any():
        raise ValueError(f"{split_name} vẫn có inf/-inf.")

print("\nFinal matrix shapes:")
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)


# ============================================================
# 8. DEFINE EVALUATION FUNCTION
# ============================================================

def evaluate_multiclass(y_true, y_pred, y_proba):
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "macro_precision": precision_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0,
        ),
        "macro_f2": fbeta_score(
            y_true,
            y_pred,
            labels=LABELS,
            beta=2,
            average="macro",
            zero_division=0,
        ),
        "weighted_precision": precision_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0,
        ),
        "weighted_recall": recall_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0,
        ),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "log_loss": log_loss(
            y_true, y_proba, labels=LABELS
        ),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    for class_name in CLASS_NAMES:
        key = class_name.lower().replace(" ", "_")
        class_metrics = report[class_name]

        result[f"{key}_precision"] = class_metrics["precision"]
        result[f"{key}_recall"] = class_metrics["recall"]
        result[f"{key}_f1"] = class_metrics["f1-score"]

        class_code = CLASS_NAMES.index(class_name)
        y_true_binary = (np.asarray(y_true) == class_code).astype(int)
        y_pred_binary = (np.asarray(y_pred) == class_code).astype(int)

        result[f"{key}_f2"] = fbeta_score(
            y_true_binary,
            y_pred_binary,
            beta=2,
            zero_division=0,
        )

    return result


# ============================================================
# 9. TRAIN LIGHTGBM + LOG TO MLFLOW
# ============================================================

# Draw receives a higher training loss weight.
# Draw receives a higher training loss weight.
sample_weight = np.where(
    y_train.to_numpy() == 1,
    DRAW_WEIGHT,
    1.0
).astype(np.float64)

assert len(sample_weight) == len(y_train)
assert np.isfinite(sample_weight).all()

print("Sample weights:")
print(f"  Normal classes: 1.0")
print(f"  Draw weight:    {DRAW_WEIGHT}")
print(f"  Draw samples:   {(y_train.to_numpy() == 1).sum():,}")
print(f"  Total samples:  {len(sample_weight):,}")

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

model = LGBMClassifier(**MODEL_PARAMS)

with mlflow.start_run(run_name=RUN_NAME) as run:
    run_id = run.info.run_id

    # ---------------------------
    # Log configuration
    # ---------------------------
    mlflow.log_params(MODEL_PARAMS)
    mlflow.log_param("draw_weight", DRAW_WEIGHT)
    mlflow.log_param("n_features", len(feature_names))
    mlflow.log_param("n_train", len(X_train))
    mlflow.log_param("n_validation", len(X_valid))
    mlflow.log_param("n_test", len(X_test))
    mlflow.log_param("class_mapping", "0=Away,1=Draw,2=Home")
    mlflow.log_param("split_method", "dbt_monthly_snapshot")
    mlflow.log_param("data_as_of", MONTHLY_WINDOW["data_as_of"])
    mlflow.log_param("window_id", MONTHLY_WINDOW["window_id"])
    mlflow.log_param("snapshot_sha256", MONTHLY_WINDOW["snapshot_sha256"])
    mlflow.log_param("valid_months", MONTHLY_WINDOW["valid_months"])
    mlflow.log_param("test_months", MONTHLY_WINDOW["test_months"])
    mlflow.set_tags({"training_policy": "monthly_v1", "deployment_status": "candidate"})
    mlflow.log_param("train_end", str(TRAIN_END.date()))
    mlflow.log_param("valid_end", str(VALID_END.date()))
    mlflow.log_param("max_missing_rate_train", MAX_MISSING_RATE)
    mlflow.log_param("date_column", DATE_COL)
    mlflow.log_param("target_column", TARGET_COL)

    for split_name, part in [
        ("train", train_df),
        ("valid", valid_df),
        ("test", test_df),
    ]:
        mlflow.log_param(
            f"{split_name}_start_date",
            str(part[DATE_COL].min().date()),
        )
        mlflow.log_param(
            f"{split_name}_end_date",
            str(part[DATE_COL].max().date()),
        )

    print("\nTraining LightGBM...")
    print("MLflow run ID:", run_id)

    model.fit(
        X_train,
        y_train,
        sample_weight=sample_weight,
    )

    # ---------------------------
    # Predictions
    # ---------------------------
    valid_proba = model.predict_proba(X_valid)
    test_proba = model.predict_proba(X_test)

    model_classes = list(model.classes_)
    if model_classes != LABELS:
        raise ValueError(
            f"Model classes {model_classes} != expected {LABELS}"
        )

    valid_pred = np.argmax(valid_proba, axis=1)
    test_pred = np.argmax(test_proba, axis=1)

    # ---------------------------
    # Metrics
    # ---------------------------
    valid_metrics = evaluate_multiclass(
        y_valid.to_numpy(), valid_pred, valid_proba
    )
    test_metrics = evaluate_multiclass(
        y_test.to_numpy(), test_pred, test_proba
    )

    print("\n" + "=" * 70)
    print("VALIDATION METRICS")
    print("=" * 70)
    print(pd.Series(valid_metrics).round(4).to_string())

    print("\nVALIDATION CLASSIFICATION REPORT")
    print(classification_report(
        y_valid,
        valid_pred,
        labels=LABELS,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))

    print("\n" + "=" * 70)
    print("TEST METRICS - DIAGNOSTIC UNTIL MODEL IS FROZEN")
    print("=" * 70)
    print(pd.Series(test_metrics).round(4).to_string())

    print("\nTEST CLASSIFICATION REPORT")
    print(classification_report(
        y_test,
        test_pred,
        labels=LABELS,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))

    # ---------------------------
    # Confusion matrices
    # ---------------------------
    cm_valid = confusion_matrix(
        y_valid, valid_pred, labels=LABELS
    )
    cm_test = confusion_matrix(
        y_test, test_pred, labels=LABELS
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ConfusionMatrixDisplay(
        confusion_matrix=cm_valid,
        display_labels=CLASS_NAMES,
    ).plot(
        ax=axes[0],
        values_format="d",
        colorbar=False,
    )
    axes[0].set_title("Validation Confusion Matrix")

    ConfusionMatrixDisplay(
        confusion_matrix=cm_test,
        display_labels=CLASS_NAMES,
    ).plot(
        ax=axes[1],
        values_format="d",
        colorbar=False,
    )
    axes[1].set_title("Test Confusion Matrix")

    plt.tight_layout()

    # ---------------------------
    # Feature importance
    # ---------------------------
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)

    print("\nTop 20 feature importance:")
    print(importance_df.head(20).to_string(index=False))

    # ---------------------------
    # Save artifacts
    # ---------------------------
    with tempfile.TemporaryDirectory() as tmpdir:

        # Metrics
        metrics_df = pd.DataFrame({
            "validation": pd.Series(valid_metrics),
            "test": pd.Series(test_metrics),
        })
        metrics_path = os.path.join(
            tmpdir, "multiclass_metrics.csv"
        )
        metrics_df.to_csv(metrics_path, index=True)
        mlflow.log_artifact(metrics_path)

        # Classification reports
        for split_name, y_true, y_pred in [
            ("validation", y_valid, valid_pred),
            ("test", y_test, test_pred),
        ]:
            report = classification_report(
                y_true,
                y_pred,
                labels=LABELS,
                target_names=CLASS_NAMES,
                output_dict=True,
                zero_division=0,
            )

            report_path = os.path.join(
                tmpdir,
                f"classification_report_{split_name}.json",
            )
            with open(report_path, "w", encoding="utf-8") as f:
                json.dump(report, f, ensure_ascii=False, indent=2)

            mlflow.log_artifact(report_path)

        # Confusion matrices
        for split_name, cm in [
            ("validation", cm_valid),
            ("test", cm_test),
        ]:
            cm_df = pd.DataFrame(
                cm,
                index=[f"true_{c}" for c in CLASS_NAMES],
                columns=[f"pred_{c}" for c in CLASS_NAMES],
            )

            cm_path = os.path.join(
                tmpdir, f"confusion_matrix_{split_name}.csv"
            )
            cm_df.to_csv(cm_path)
            mlflow.log_artifact(cm_path)

        # Predictions and probabilities
        for split_name, part, y_true, y_pred, y_proba in [
            (
                "validation",
                valid_df,
                y_valid.to_numpy(),
                valid_pred,
                valid_proba,
            ),
            (
                "test",
                test_df,
                y_test.to_numpy(),
                test_pred,
                test_proba,
            ),
        ]:
            prediction_columns = {}

            if KEY_COL in part.columns:
                prediction_columns[KEY_COL] = part[KEY_COL].to_numpy()

            prediction_columns[DATE_COL] = part[DATE_COL].to_numpy()
            prediction_columns["y_true"] = y_true
            prediction_columns["y_pred"] = y_pred
            prediction_columns["p_away"] = y_proba[:, 0]
            prediction_columns["p_draw"] = y_proba[:, 1]
            prediction_columns["p_home"] = y_proba[:, 2]

            pred_df = pd.DataFrame(prediction_columns)

            pred_path = os.path.join(
                tmpdir, f"predictions_{split_name}.csv"
            )
            pred_df.to_csv(pred_path, index=False)
            mlflow.log_artifact(pred_path)

        # Feature importance and feature names
        importance_path = os.path.join(
            tmpdir, "feature_importance.csv"
        )
        importance_df.to_csv(importance_path, index=False)
        mlflow.log_artifact(importance_path)

        features_path = os.path.join(
            tmpdir, "feature_names.json"
        )
        with open(features_path, "w", encoding="utf-8") as f:
            json.dump(feature_names, f, ensure_ascii=False, indent=2)
        mlflow.log_artifact(features_path)

        # Data and training metadata
        metadata = {
            "run_id": run_id,
            "date_column": DATE_COL,
            "target_column": TARGET_COL,
            "class_mapping": CLASS_MAPPING,
            "train_end": str(TRAIN_END.date()),
            "valid_end": str(VALID_END.date()),
            "monthly_window": MONTHLY_WINDOW,
            "max_missing_rate_train": MAX_MISSING_RATE,
            "draw_weight": DRAW_WEIGHT,
            "n_raw": int(n_raw),
            "n_after_filter": int(len(data)),
            "n_train": int(len(X_train)),
            "n_validation": int(len(X_valid)),
            "n_test": int(len(X_test)),
            "n_features": int(len(feature_names)),
            "train_missing_rate_threshold": MAX_MISSING_RATE,
        }

        window_path = os.path.join(tmpdir, "monthly_window.json")
        with open(window_path, "w", encoding="utf-8") as f:
            json.dump(MONTHLY_WINDOW, f, ensure_ascii=False, indent=2)
        mlflow.log_artifact(window_path)

        metadata_path = os.path.join(
            tmpdir, "training_metadata.json"
        )
        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)
        mlflow.log_artifact(metadata_path)

        # Confusion matrix figure
        fig_path = os.path.join(
            tmpdir, "confusion_matrices.png"
        )
        fig.savefig(fig_path, dpi=160, bbox_inches="tight")
        mlflow.log_artifact(fig_path)

        # Log metrics
        all_mlflow_metrics = {
            **{
                f"valid_{k}": float(v)
                for k, v in valid_metrics.items()
            },
            **{
                f"test_{k}": float(v)
                for k, v in test_metrics.items()
            },
        }
        mlflow.log_metrics(all_mlflow_metrics)

        # Log LightGBM model
        mlflow.lightgbm.log_model(
            model,
            artifact_path="model",
        )

    plt.show()
    plt.close(fig)

    print("\n" + "=" * 70)
    print("RUN COMPLETED")
    print("Experiment:", EXPERIMENT_NAME)
    print("Run name:", RUN_NAME)
    print("Run ID:", run_id)
    print("Tracking URI:", TRACKING_URI)
    print("Feature count:", len(feature_names))
    print(f"Validation Macro F1: {valid_metrics['macro_f1']:.4f}")
    print(f"Validation Macro F2: {valid_metrics['macro_f2']:.4f}")
    print(f"Validation Draw Recall: {valid_metrics['draw_recall']:.4f}")
    print("=" * 70)
